# 06 · LoRA fine-tuning of the embedding model

Trains a LoRA adapter (rank 16 on the query/key/value projections, ~2.4 M trainable parameters out of 335 M) for `mixedbread-ai/mxbai-embed-large-v1` on (query, positive plot) pairs from notebook 05 with `MultipleNegativesRankingLoss`. The base weights stay frozen, which is what prevented the collapse seen in notebook 04.

Configured for a fast CPU run: 500 pairs, 1 epoch, batch size 4, max sequence length 192. `models/lora_v1` and `models/lora_v2` were produced with this notebook.

After training, re-encode the corpus with the adapter in notebook 03 and evaluate in notebook 08. Result of LoRA v2 on the 394-query test set: Hit@1 39.6 %, Hit@10 56.6 %, MRR 0.445 (base model: 37.3 % / 52.0 % / 0.415).

In [ ]:
import os

# Must be set before importing torch/transformers so the kernel never initializes CUDA.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "0"

from pathlib import Path
import random

import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

from peft import LoraConfig, get_peft_model
from sentence_transformers import InputExample, SentenceTransformer, losses


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd().parent
TRAINING_CSV = PROJECT_DIR / "data" / "training" / "training_pairs_hard_negatives.csv"
OUTPUT_DIR = PROJECT_DIR / "models" / "lora_new"  # rename to lora_vN once you keep it
FINAL_DIR = OUTPUT_DIR

MODEL_NAME = "mixedbread-ai/mxbai-embed-large-v1"
MAX_SEQ_LENGTH = 192
MAX_TRAIN_SAMPLES = 500
BATCH_SIZE = 4
NUM_EPOCHS = 1
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
USE_CPU = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import torch

if USE_CPU:
    # Keep transformers/trainer setup from touching a stale CUDA context in this notebook session.
    torch.cuda.is_available = lambda: False
    torch.cuda.device_count = lambda: 0
    torch.cuda.manual_seed_all = lambda *args, **kwargs: None
    torch.cuda.manual_seed = lambda *args, **kwargs: None

print({
    "use_cpu": USE_CPU,
    "torch_cuda_available": torch.cuda.is_available(),
    "torch_cuda_device_count": torch.cuda.device_count(),
})

In [ ]:
def add_query_prefix(query: str) -> str:
    return f"Represent this sentence for searching relevant passages: {str(query).strip()}"


def normalize_text(value: str) -> str:
    return str(value).strip()


df = pd.read_csv(TRAINING_CSV)
df = df.dropna(subset=["query", "positive"]).copy()

# Fast mode: one query-positive pair per row. The hard-negative mining still helped produce these rows,
# but we train with in-batch negatives for speed.
pairs_df = pd.DataFrame({
    "query": df["query"].map(add_query_prefix),
    "positive": df["positive"].map(normalize_text),
}).drop_duplicates().reset_index(drop=True)

if MAX_TRAIN_SAMPLES is not None:
    pairs_df = pairs_df.sample(min(MAX_TRAIN_SAMPLES, len(pairs_df)), random_state=SEED)
    pairs_df = pairs_df.reset_index(drop=True)

pairs_df.head()

In [ ]:
print(f"Source rows: {len(df)}")
print(f"Training pairs used: {len(pairs_df)}")
pairs_df.isna().sum()

In [ ]:
model = SentenceTransformer(MODEL_NAME, device="cpu" if USE_CPU else None)
model.max_seq_length = MAX_SEQ_LENGTH

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "key", "value"],
)

model._first_module().auto_model = get_peft_model(
    model._first_module().auto_model,
    lora_config,
)

model._first_module().auto_model.print_trainable_parameters()

In [ ]:
train_examples = [
    InputExample(texts=[row.query, row.positive])
    for row in pairs_df.itertuples(index=False)
]

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
)

train_loss = losses.MultipleNegativesRankingLoss(model)

print({
    "train_examples": len(train_examples),
    "batch_size": BATCH_SIZE,
    "steps_per_epoch": len(train_dataloader),
    "epochs": NUM_EPOCHS,
})

In [ ]:
# Run this cell only when you are ready to train.
# This is the simplest fast version: one epoch, capped samples, no evaluator, no checkpoints.

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=NUM_EPOCHS,
    warmup_steps=max(1, int(len(train_dataloader) * WARMUP_RATIO)),
    optimizer_params={"lr": LEARNING_RATE},
    show_progress_bar=True,
)

model.save(str(FINAL_DIR))
